# STEP 9 — Signature finding: predictability ≠ profitability

No NEW experiment. Step 2 PR-AUC (LightGBM) + Step 5 EMP + churner CLV distribution.
Hypothesis: high-PR-AUC set (iranian) has narrow/low churner value → low EMP; low-PR-AUC
(telco) has wide/high → high EMP. n=5 is small → trend/illustration. The heavy logic lives in `src/signature.py`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


def _bul_kok():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "config.yaml").exists():
            return c
    raise RuntimeError("config.yaml not found")


KOK = _bul_kok()
if str(KOK) not in sys.path:
    sys.path.insert(0, str(KOK))

warnings.filterwarnings("ignore")
from src import config as cfg
from src import plotstyle as ps
from src import signature as sg
from src import strings as S

ps.uygula()
cfg.klasorleri_hazirla()
pd.set_option("display.width", 200)
CIKTI = []


def yaz(s=""):
    print(s)
    CIKTI.append(str(s))


veriler = {k: pd.read_csv(cfg.PROCESSED / f"{k}_clean.csv") for k in cfg.DATASETS}

## B1+B2 — Evidence table (PR-AUC, EMP, churner CLV median/Gini/CV, churn rate)

In [2]:
df, churner_clv, prauc, emp = sg.kanit_tablosu(veriler)
yaz(S.MSG["bolum"].format(ad="SIGNATURE EVIDENCE"))
yaz(df.to_string(index=False))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "signature_evidence.csv"))

===== SIGNATURE EVIDENCE =====
  Dataset  PR-AUC    EMP  Churner CLV median  Churner CLV Gini  Churner CLV CV  Churn rate
    telco  0.6635 0.0491              1911.6             0.184           0.331       0.265
cell2cell  0.4661 0.0352              1146.7             0.345           0.766       0.288
ecommerce  0.9075 0.0162              3584.2             0.123           0.237       0.171
  iranian  0.9576 0.0007                96.8             0.502           1.036       0.157
     bank  0.7065 0.0319              4374.0             0.350           0.640       0.204
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/tables/signature_evidence.csv


## B1 — Scatter: PR-AUC vs EMP (correlation)

In [3]:
rho, pr_, r, pp = sg.korelasyon(prauc, emp)
yaz(S.MSG9["korelasyon"].format(rho=rho, pr=pr_, r=r, pp=pp))
y1 = sg.figur_scatter(prauc, emp)
yaz(S.MSG["kayit"].format(yol=y1))

PR-AUC↔EMP: Spearman ρ=-0.900 (p=0.037), Pearson r=-0.787 (p=0.114). n=5 small → trend/illustration, no strong claim.
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_signature/prauc_vs_emp_scatter.png


## B2 — Churner value distributions (5 panels)

In [4]:
y2 = sg.figur_dagilim(churner_clv)
yaz(S.MSG["kayit"].format(yol=y2))

Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_signature/churner_value_distributions.png


## B3 — PR-AUC ranking vs EMP ranking (inverse?)

In [5]:
yaz(S.MSG["bolum"].format(ad="RANKING COMPARISON"))
pr_sira = sorted(cfg.DATASETS, key=lambda s: prauc[s], reverse=True)
emp_sira = sorted(cfg.DATASETS, key=lambda s: emp[s], reverse=True)
yorum = "rankings are INVERSE/unrelated -> predictability does not guarantee profitability" \
    if pr_sira != emp_sira else "rankings coincide"
yaz(S.MSG9["sira"].format(prs=" > ".join(pr_sira), emps=" > ".join(emp_sira), yorum=yorum))
yaz(f"\nExample: iranian PR-AUC={prauc['iranian']:.2f} (highest) but EMP={emp['iranian']:.4f} (lowest); "
    f"telco PR-AUC={prauc['telco']:.2f} (lowest) but EMP={emp['telco']:.4f} (highest).")
yaz(S.MSG9["bitti"])

_log = cfg.LOGS / "adim9_ozet.log"
_log.write_text("\n".join(CIKTI) + "\n", encoding="utf-8")
print(S.MSG["kayit"].format(yol=_log))

===== RANKING COMPARISON =====
PR-AUC ranking: iranian > ecommerce > bank > telco > cell2cell  |  EMP ranking: telco > cell2cell > bank > ecommerce > iranian  -> rankings are INVERSE/unrelated -> predictability does not guarantee profitability

Example: iranian PR-AUC=0.96 (highest) but EMP=0.0007 (lowest); telco PR-AUC=0.66 (lowest) but EMP=0.0491 (highest).
STEP 9 complete. Signature finding (predictability ≠ profitability) documented.
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/logs/adim9_ozet.log
